# Find corrupt camera footage in a robot dataset

[`zima-robot/liftplace-ee-xyz-500-v2`](https://huggingface.co/datasets/zima-robot/liftplace-ee-xyz-500-v2), 500 simulated pick-and-place episodes, graded by Kalanos 0.10.2 at revision `d796a5d6`.
The notebook charts what the grade found from a saved copy of it, and downloads only the video of the episodes it shows.
CPU only, no API key, nothing is published. Choose **Runtime → Run all**.

These are recorded audit decisions, not task-success or model-performance measurements.

## Install

Kalanos with video checks, and two helpers, at pinned versions.

In [ ]:
# @title Install Kalanos
%pip install -q --no-warn-conflicts "kalanos[numeric,hf,video]==0.10.2" huggingface-hub==1.33.0 plotly==7.1.0

## Load the saved grade

The grade is the output of these commands, run on the pinned dataset with the same pins as the install cell:

```bash
hf download zima-robot/liftplace-ee-xyz-500-v2 --repo-type dataset --revision d796a5d6438a2d72f714fde611e7a429650d2c72 --local-dir liftplace-ee-xyz-500-v2
kalanos grade liftplace-ee-xyz-500-v2 --report liftplace.json
```

The full report is about 186 MB, so the notebook saves only the parts it charts: each episode's decision, score and findings, and each camera's measurements, the frames Kalanos flagged and the video file.
The last section runs the grade again. It downloads about 451 MB and takes about 9 minutes on a CPU runtime with two cores, like a free Colab one.

In [ ]:
# @title Download and load the saved grade
import json
from pathlib import Path
from urllib.request import urlretrieve

import polars as pl


DATASET_ID = "zima-robot/liftplace-ee-xyz-500-v2"
# A fixed revision keeps the files, and so every result below, the same on each run.
REVISION = "d796a5d6438a2d72f714fde611e7a429650d2c72"
DATASET = Path("liftplace-ee-xyz-500-v2")
CAMERAS = ("agentview", "wrist")
# The parts of the Kalanos report the charts need, pinned to one commit.
# The last section produces them again from the dataset.
SAVED_URL = "https://raw.githubusercontent.com/KalanosAI/kalanos/79fcc15f96adb5625a4c2094ad9d05f27a86a42a/notebooks/reports/liftplace_findings.json"

urlretrieve(SAVED_URL, "liftplace_findings.json")
summary = json.loads(Path("liftplace_findings.json").read_text())

# Kalanos grades each camera of an episode separately, and one corrupt camera blocks the episode,
# so the episode table keeps the worse of the two cameras.
cameras = pl.DataFrame(
    [
        {
            "episode": e["index"],
            "segment": e["segment"],
            "decision": e["decision"],
            "findings": ", ".join(sorted({metric for metric, _ in e["reasons"]})),
            "data_frames": e["data_frames"],
            "camera": camera,
        }
        | e["cameras"][camera]
        for e in summary["episodes"]
        for camera in CAMERAS
    ]
)
episodes = cameras.group_by("episode", maintain_order=True).agg(
    pl.first("segment", "decision", "findings"),
    corrupt_pct=pl.max("corrupt_frame_pct"),
)

source, counts = summary["source"], summary["counts"]
print(
    f"{DATASET_ID} @ {REVISION[:8]}, Kalanos {source['kalanos']}, {source['profile']}:"
    f" {counts['eligible']} eligible, {counts['review']} review,"
    f" {counts['blocked']} blocked of {len(episodes)} episodes"
)

## What Kalanos flagged

`corrupt_frame_pct` is the share of a camera's examined frames that are blank or hold decoder noise ([definition](https://github.com/KalanosAI/kalanos/blob/v0.10.2/docs/METRICS.md?plain=1#L210)). Kalanos blocks an episode from 10%, in every profile.
At the default tier it examines 10 evenly spaced frames per camera in each episode, so an episode with only a few bad frames could pass. With `--tier full`, it reads them all.
Each of the 8 runs of corrupt episodes ends on the last episode of a recording session (the dataset's `collection_segment_index` column), so the footage stays broken until the next session starts.

In [ ]:
# @title Chart the corrupt frames by episode
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

import kalanos


INK, NAVY, MUTED, RULE = "#171717", "#173044", "#65625C", "#D2CEC5"
COLORS = {"pass": "#23735E", "review": "#87511B", "blocked": "#A62B23"}
ORDER = {"decision": list(COLORS)}
axis = {"gridcolor": RULE, "linecolor": "#C8C3B8", "zeroline": False, "ticks": ""}
pio.templates["kalanos"] = go.layout.Template(
    layout={
        "font": {
            "family": "IBM Plex Sans, Arial, sans-serif",
            "size": 14,
            "color": INK,
        },
        "title": {
            "x": 0,
            "xref": "paper",
            "font": {"size": 20, "color": NAVY, "weight": 600},
            "subtitle": {
                "font": {
                    "family": "IBM Plex Mono, monospace",
                    "size": 12,
                    "color": MUTED,
                }
            },
        },
        "paper_bgcolor": "#F4F1EA",
        "plot_bgcolor": "#FFFFFF",
        "xaxis": axis,
        "yaxis": axis,
        "margin": {"t": 110, "r": 24, "b": 56, "l": 64},
    }
)
pio.templates.default = "plotly_white+kalanos"
# A Kalanos result holds for the dataset revision, Kalanos version, profile and frames it examined,
# so each chart states them.
SCOPE = (
    f"{DATASET_ID} @ {REVISION[:8]} | Kalanos {source['kalanos']} | {source['profile']}"
    f" | {summary['frames_decoded']:,} of {summary['frames_total']:,} camera frames examined"
)

# The bound at which Kalanos blocks an episode comes from its default policy, the one the grade used.
policy = kalanos.load_policy(None)
corrupt_bound = policy.metrics["vision.corrupt_frame_pct"].thresholds["default"].bad
corrupt = episodes.filter(pl.col("corrupt_pct") >= corrupt_bound)

fig = px.bar(
    episodes,
    x="episode",
    y="corrupt_pct",
    color="decision",
    color_discrete_map=COLORS,
    category_orders=ORDER,
    hover_data=["segment", "findings"],
    labels={
        "episode": "Episode",
        "corrupt_pct": "Corrupt frames (% examined)",
    },
    title=f"{corrupt.height} of {episodes.height} episodes have corrupt camera frames",
    subtitle=SCOPE,
)
fig.add_hline(
    y=corrupt_bound,
    line={"dash": "dash", "color": COLORS["blocked"], "width": 1},
    annotation_text=f"{corrupt_bound:g}% block bound",
    annotation_position="top left",
    annotation_font_color=COLORS["blocked"],
)
fig.update_traces(marker_line_width=0)
fig.update_yaxes(range=[0, 105])
fig.update_layout(bargap=0, showlegend=False)
fig.show()

## Video longer than its data

`frame_count_vs_timebase` compares the frames in an episode's video with the rows of its data ([definition](https://github.com/KalanosAI/kalanos/blob/v0.10.2/docs/METRICS.md?plain=1#L214)). Kalanos blocks from a difference of 5%.

In [ ]:
# @title Compare video frames with data rows where they differ
# `frame_count_vs_timebase` is the gap between the frames in an episode's video and the rows of its data,
# as a fraction of the rows.
frame_bound = policy.metrics["vision.frame_count_vs_timebase"].thresholds["default"].bad
longer = (
    cameras.filter(pl.col("frame_count_vs_timebase") >= frame_bound)
    .with_columns(stream=pl.format("Episode {} {}", "episode", "camera"))
    .unpivot(["data_frames", "video_frames"], index="stream", variable_name="count")
)
names = longer["stream"].str.extract(r"Episode (\d+)").cast(pl.Int64).unique().sort()

fig = px.bar(
    longer,
    x="stream",
    y="value",
    color="count",
    barmode="group",
    text="value",
    color_discrete_map={"data_frames": NAVY, "video_frames": COLORS["blocked"]},
    labels={"stream": "", "value": "Frames", "count": ""},
    title=f"The video of episodes {' and '.join(map(str, names))} runs past the end of its data",
    subtitle=SCOPE,
)
legend = {"data_frames": "rows in the data", "video_frames": "frames in the video"}
fig.for_each_trace(lambda t: t.update(name=legend[t.name]))
fig.update_layout(height=440)
fig.show()

## Find and see the errors

Every episode that did not pass, with what Kalanos found on each camera. The episode number opens the episode in the [LeRobot dataset visualizer](https://huggingface.co/spaces/lerobot/visualize_dataset), which shows the dataset's current revision, and the metric links to its definition in the Kalanos 0.10.2 docs.

In [ ]:
# @title List every affected episode
from urllib.parse import quote

from IPython.display import Markdown, display


VISUALIZER = "https://huggingface.co/spaces/lerobot/visualize_dataset?path="
DOCS = "https://github.com/KalanosAI/kalanos/blob/v0.10.2/docs/METRICS.md?plain=1#L"
# The line of each metric's definition in the Kalanos 0.10.2 docs.
DOC_LINES = {
    "corrupt_frame_pct": 210,
    "frame_count_vs_timebase": 214,
    "flatline_pct": 87,
}


def visualizer(index: int) -> str:
    """Return the LeRobot visualizer link of an episode."""

    return VISUALIZER + quote(f"/{DATASET_ID}/episode_{index}", safe="")


def found(episode: dict) -> str:
    """Return the metrics that decided an episode, each linked to its definition, with where they were measured."""

    sources = {}
    for metric, source in episode["reasons"]:
        sources.setdefault(metric, []).append(
            source.removeprefix("observation.images.")
        )
    return "; ".join(
        f"[`{metric}`]({DOCS}{DOC_LINES[metric]}) ({', '.join(where)})"
        for metric, where in sorted(sources.items())
    )


def evidence(episode: dict, camera: str) -> str:
    """Return what Kalanos measured on one camera of an episode, in words."""

    video = episode["cameras"][camera]
    if video["frame_count_vs_timebase"] >= frame_bound:
        return f"{video['video_frames']} video frames for {episode['data_frames']} data rows"
    if video["flagged_frames"]:
        return f"{len(video['flagged_frames'])} of {video['frames_examined']} examined frames corrupt"
    return ""


lines = [
    "| Episode | Decision | What Kalanos found | agentview | wrist |",
    "| --- | --- | --- | --- | --- |",
]
affected = [e for e in summary["episodes"] if e["decision"] != "pass"]
for e in sorted(affected, key=lambda e: (e["decision"], e["index"])):
    cells = " | ".join(evidence(e, c) for c in CAMERAS)
    lines.append(
        f"| [{e['index']}]({visualizer(e['index'])}) | {e['decision']} | {found(e)} | {cells} |"
    )
display(Markdown("\n".join(lines)))

For each episode in `EPISODES`, the frames Kalanos flagged as corrupt, or, for a video longer than its data, the frames past the end of the data, then the video of each camera.
When a camera has fewer than six flagged frames, the strip fills up with frames Kalanos did not flag, labelled as such.
The default episodes show the three kinds of error: about half of the examined frames noise (146), every examined frame blank (149) or noise (282), and a video longer than its data (92). Any episode number from the table above works.

In [ ]:
# @title See the errors in chosen episodes
import av
import numpy as np
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import disable_progress_bars
from IPython.display import Video


EPISODES = [146, 149, 282, 92]  # @param {type:"raw"}
STRIP_FRAMES = 6

disable_progress_bars()


def read_frames(path: str, start_s: float, end_s: float | None) -> list[np.ndarray]:
    """Decode the frames of one episode from a video file.

    Parameters
    ----------
    path
        The video file.
    start_s
        Where the episode starts in the file, in seconds.
    end_s
        Where it ends, or `None` to read to the end of the file.

    Returns
    -------
    list of np.ndarray
        The frames in order, as RGB arrays.
    """

    with av.open(path) as container:
        stream = container.streams.video[0]
        half = 0.5 / float(stream.average_rate or 20)
        times = (start_s - half, float("inf") if end_s is None else end_s - half)
        return [
            frame.to_ndarray(format="rgb24")
            for frame in container.decode(stream)
            if frame.time is not None and times[0] <= frame.time < times[1]
        ]


def spread(indices: list[int], count: int) -> list[int]:
    """Return up to `count` evenly spaced items of `indices`, in order."""

    if len(indices) <= count:
        return list(indices)
    return [indices[round(i)] for i in np.linspace(0, len(indices) - 1, count)]


def frames_to_show(
    video: dict, n_frames: int, data_frames: int
) -> list[tuple[int, str]]:
    """Choose the frames of one camera that show the error, each with its label.

    Parameters
    ----------
    video
        The camera's entry in the saved grade.
    n_frames
        How many frames the camera's video holds for the episode.
    data_frames
        How many rows the episode's data holds.

    Returns
    -------
    list of tuple of int and str
        Frame index and label, in frame order.
    """

    # Frames past the data are the ones `frame_count_vs_timebase` counted.
    if n_frames > data_frames:
        inside = spread(list(range(data_frames)), STRIP_FRAMES // 2)
        past = spread(list(range(data_frames, n_frames)), STRIP_FRAMES - len(inside))
        return [(i, "") for i in inside] + [(i, "past the data") for i in past]
    # The frames Kalanos examined and found blank or noisy.
    flagged = spread(video["flagged_frames"], STRIP_FRAMES)
    others = [i for i in range(n_frames) if i not in video["flagged_frames"]]
    rest = spread(others, STRIP_FRAMES - len(flagged))
    shown = [(i, "flagged corrupt") for i in flagged] + [
        (i, "not flagged") for i in rest
    ]
    return sorted(shown)


by_index = {e["index"]: e for e in summary["episodes"]}
for index in EPISODES:
    episode = by_index[index]
    display(
        Markdown(
            f"### Episode {index}: {episode['decision']}\n\n"
            f"{found(episode) or 'Nothing flagged.'} "
            f"[Open it in the LeRobot visualizer]({visualizer(index)})."
        )
    )
    images, titles, videos = [], [], []
    for camera in CAMERAS:
        video = episode["cameras"][camera]
        path = hf_hub_download(
            repo_id=DATASET_ID,
            repo_type="dataset",
            revision=REVISION,
            filename=video["video_file"],
            local_dir=DATASET,
        )
        frames = read_frames(path, video["from_s"], video["to_s"])
        for i, label in frames_to_show(video, len(frames), episode["data_frames"]):
            images.append(frames[i])
            titles.append(
                f"{camera} frame {i}" + (f"<br><b>{label}</b>" if label else "")
            )
        videos.append(path)
    fig = px.imshow(
        np.stack(images),
        facet_col=0,
        facet_col_wrap=STRIP_FRAMES,
        facet_row_spacing=0.14,
        facet_col_spacing=0.01,
        binary_string=True,
    )
    for annotation in fig.layout.annotations:
        annotation.update(
            text=titles[int(annotation.text.split("=")[1])], font_color=MUTED
        )
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    fig.update_layout(height=440, margin={"t": 50, "b": 10, "l": 10, "r": 10})
    fig.show()
    for camera, path in zip(CAMERAS, videos, strict=True):
        display(Markdown(f"{camera} video, episode {index}"))
        display(Video(path, embed=True, width=256, html_attributes="controls loop"))

## Recreate the results

Optional. Tick `REGRADE` to run the commands from *Load the saved grade* here: the cell downloads the dataset, grades it with the default tier and profile, and checks every episode's decision and findings against the saved grade. The charts above keep the saved grade.
It downloads about 451 MB and takes about 9 minutes on a CPU runtime with two cores, and about 2 GB of memory.
The saved grade itself comes from [`summarize_liftplace.py`](https://github.com/KalanosAI/kalanos/blob/6e5c13d8735f05871cfde8b7fbce3e6a3a88edf6/notebooks/reports/summarize_liftplace.py), which grades the same local copy and keeps the parts above:

```bash
uv run notebooks/reports/summarize_liftplace.py liftplace-ee-xyz-500-v2 liftplace_findings.json
```

In [ ]:
# @title Regrade and compare
import logging

from huggingface_hub import snapshot_download

import kalanos


REGRADE = False  # @param {type:"boolean"}


def decisions(report: kalanos.Report) -> dict[int, tuple]:
    """Return each episode's decision and the findings behind it, in the saved grade's form."""

    reasons = {}
    for f in report.findings:
        if f.consequence.value in ("review", "block"):
            source = "/".join(filter(None, [f.source_field, f.channel]))
            reasons.setdefault(f.episode_id, set()).add(
                (f.metric_id.split(".", 1)[1], source)
            )
    return {
        int(e.id.rsplit("_", 1)[1]): (
            e.eligibility.status.value if e.eligibility else "unknown",
            [list(r) for r in sorted(reasons.get(e.id, ()))],
        )
        for e in report.episodes
    }


if not REGRADE:
    print("Regrade off. Tick REGRADE to grade the dataset again.")
else:
    snapshot_download(
        repo_id=DATASET_ID, repo_type="dataset", revision=REVISION, local_dir=DATASET
    )
    # As the CLI's `--verbosity error`: hides warnings about file formats this dataset doesn't use.
    logging.getLogger("kalanos").setLevel(logging.ERROR)
    print(f"Grading {DATASET_ID}...", flush=True)
    # `kalanos.grade` is the Python form of `kalanos grade`, with the same default tier and profile as the saved grade.
    fresh = kalanos.grade(DATASET)
    saved = {e["index"]: (e["decision"], e["reasons"]) for e in summary["episodes"]}
    differ = sorted(i for i, d in decisions(fresh).items() if saved.get(i) != d)
    version = fresh.producer.version if fresh.producer else "unknown"
    verdict = (
        f"decisions or findings differ from the saved grade in episodes {differ}"
        if differ
        else "same decisions and findings as the saved grade"
    )
    print(f"{DATASET_ID}: Kalanos {version}, {verdict}")

**Continue with Kalanos:** [Browse dataset reports](https://hub.kalanos.ai) · [Use the grader locally](https://github.com/KalanosAI/kalanos) · [Attach an audit to a training run](https://github.com/KalanosAI/kalanos-tracking)